# Q2 - Lexical Candidate Generation (BM25)

Builds two independent from-scratch BM25 indexes (one per dataset, via
`cs4406m26_assignment1c1.bm25`) over the unified `articles` feature store from
Q1, retrieves top-K candidates per user from their pre-window click history,
and reports recall@K per SPEC.md's Q2 section.

Run top-to-bottom (or via `python bm25_retrieval.py`) to rebuild
`data/processed/{dataset}/bm25_topk.parquet` and `bm25_metrics.json`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, top_k


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
DATASETS = ["ebnerd", "mind"]

RECENT_N_CLICKS = 20
BM25_K1 = 1.5
BM25_B = 0.75
CANDIDATE_K_VALUES = [50, 100, 200]
TOPK_MAX = max(CANDIDATE_K_VALUES)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

feature_store = {
    name: {
        "articles": pd.read_parquet(DATA_DIR / name / "articles.parquet"),
        "behaviors": pd.read_parquet(DATA_DIR / name / "behaviors.parquet"),
        "history": pd.read_parquet(DATA_DIR / name / "history.parquet"),
    }
    for name in DATASETS
}

{name: {table: df.shape for table, df in tables.items()} for name, tables in feature_store.items()}

{'ebnerd': {'articles': (11777, 8),
  'behaviors': (50080, 8),
  'history': (1935, 6)},
 'mind': {'articles': (65238, 8),
  'behaviors': (230117, 8),
  'history': (94057, 6)}}

## Tokenizer

`tokenize` is imported from `cs4406m26_assignment1c1.bm25` (lowercase +
Unicode-aware `\w+` splitting - works for both Danish and English without a
new dependency, since Python's `re` treats `\w` as Unicode by default, so
`æøå`/`ÆØÅ` stay intact as word characters). This is a smoke test of the
imported function, not a redefinition.

In [2]:
def test_tokenize():
    assert tokenize(None) == []
    assert tokenize("") == []
    assert tokenize("ÆØÅ æble Rødgrød") == ["æøå", "æble", "rødgrød"]
    assert tokenize("Harry's dna-test") == ["harry", "s", "dna", "test"]


test_tokenize()
print("ok: tokenizer handles Danish/English text and null/empty input")

ok: tokenizer handles Danish/English text and null/empty input


## Per-dataset tokenized corpus (title + abstract)

`abstract` must be `.fillna("")`'d before concatenation - ~5.2% of MIND
articles have a null abstract, and naive concatenation would propagate NaN and
silently zero out those articles' tokens.

In [3]:
corpus = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    text = articles["title"].fillna("") + " " + articles["abstract"].fillna("")
    doc_tokens = text.map(tokenize).tolist()
    corpus[name] = {
        "doc_ids": articles["article_id"].to_numpy(),
        "doc_tokens": doc_tokens,
        "doc_len": np.array([len(t) for t in doc_tokens]),
    }

{name: {"n_docs": len(c["doc_tokens"]), "avg_doc_len": c["doc_len"].mean()} for name, c in corpus.items()}

{'ebnerd': {'n_docs': 11777, 'avg_doc_len': np.float64(25.803685149019273)},
 'mind': {'n_docs': 65238, 'avg_doc_len': np.float64(47.23055581103038)}}

In [4]:
def test_corpus_alignment():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        c = corpus[name]
        assert len(c["doc_tokens"]) == len(articles)
        assert (c["doc_ids"] == articles["article_id"].to_numpy()).all()

    # MIND has null abstracts (~5.2%) -- must not silently zero out those articles' tokens
    mind_articles = feature_store["mind"]["articles"]
    null_abstract_positions = np.flatnonzero(mind_articles["abstract"].isna().to_numpy())
    assert len(null_abstract_positions) > 0, "expected some MIND articles with null abstract"
    sample_pos = null_abstract_positions[0]
    assert len(corpus["mind"]["doc_tokens"][sample_pos]) > 0, "null-abstract article lost its title tokens"


test_corpus_alignment()
print("ok: tokenized corpus aligned with articles and null-abstract rows still have tokens")

ok: tokenized corpus aligned with articles and null-abstract rows still have tokens


## Build a BM25Index per dataset

From-scratch postings-list inverted index (`cs4406m26_assignment1c1.bm25.build_index`):
`term -> (doc_idx array, tf array)`, plus `idf`, `doc_len`, `avgdl` computed
directly from the tokenized corpus - no third-party BM25 library. See SPEC.md
Q2 section 1 for why this is hand-built rather than a library call
(`rank_bm25`'s own scoring measured at ~2.9s/query on MIND vs. ~5ms/query
here - a library was evaluated and rejected on performance grounds).

In [5]:
bm25_index = {
    name: build_index(corpus[name]["doc_ids"], corpus[name]["doc_tokens"], k1=BM25_K1, b=BM25_B)
    for name in DATASETS
}

for name in DATASETS:
    idx = bm25_index[name]
    print(f"{name}: {idx.n_docs} docs, avgdl={idx.avgdl:.2f}, vocab={len(idx.idf)} terms")

ebnerd: 11777 docs, avgdl=25.80, vocab=31642 terms
mind: 65238 docs, avgdl=47.23, vocab=61000 terms


In [6]:
def test_bm25_index_matches_corpus():
    for name in DATASETS:
        idx = bm25_index[name]
        c = corpus[name]
        assert idx.n_docs == len(c["doc_tokens"])
        assert np.isclose(idx.avgdl, c["doc_len"].mean())
        assert np.array_equal(idx.doc_len, c["doc_len"])
        assert all(v >= 0 for v in idx.idf.values()), "non-negative IDF variant must hold for every term"

        # by-hand IDF check for a couple of real terms
        n_docs = idx.n_docs
        for term, (doc_idx, _tf) in list(idx.postings.items())[:5]:
            df = len(doc_idx)
            expected_idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
            assert np.isclose(idx.idf[term], expected_idf)


test_bm25_index_matches_corpus()
print("ok: BM25Index (doc_len/avgdl/idf/postings) matches our own tokenized corpus")

ok: BM25Index (doc_len/avgdl/idf/postings) matches our own tokenized corpus


## Top-K retrieval

Candidate generation retrieves from the whole article catalog (not a
re-ranking of the impression's given `article_ids_inview`).

In [7]:
def top_k_candidates(query_tokens: list[str], dataset: str, k: int = TOPK_MAX) -> list[tuple[str, float]]:
    return top_k(bm25_index[dataset], query_tokens, k)

In [8]:
def test_top_k_candidates():
    for name in DATASETS:
        sample_title = feature_store[name]["articles"]["title"].iloc[0]
        q = tokenize(sample_title)
        top200 = top_k_candidates(q, name, k=200)
        top50 = top_k_candidates(q, name, k=50)
        assert len(top200) == 200
        assert len(top50) == 50
        scores200 = [s for _, s in top200]
        assert scores200 == sorted(scores200, reverse=True)
        assert top200[:50] == top50, "top-50 must be a prefix of top-200 (nesting invariant)"
        assert top_k_candidates([], name, k=50) == []


test_top_k_candidates()
print("ok: top-K retrieval is sorted, nested across K, and empty for an empty query")

ok: top-K retrieval is sorted, nested across K, and empty for an empty query


## Query construction from history

Concatenates the titles of a user's most recent `RECENT_N_CLICKS` clicks.
`timestamp_sequence` is chronologically ascending, so "most recent N" is the
*last* N elements of `article_id_sequence`, not the first N.

In [9]:
def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


title_by_id = {
    name: dict(zip(feature_store[name]["articles"]["article_id"], feature_store[name]["articles"]["title"]))
    for name in DATASETS
}

In [10]:
def test_build_user_query_tokens():
    assert build_user_query_tokens([], {}) == []

    lookup = {"a": "OLDWORD", "b": "OLDWORD2", "c": "RECENTWORD"}
    ids = ["a", "b", "c"]
    assert build_user_query_tokens(ids, lookup, recent_n=1) == ["recentword"]
    tokens_all = build_user_query_tokens(ids, lookup, recent_n=10)
    assert "oldword" in tokens_all and "recentword" in tokens_all


test_build_user_query_tokens()
print("ok: query construction takes the most recent N clicks (last N, not first N)")

ok: query construction takes the most recent N clicks (last N, not first N)


## Per-user retrieval cache (val/test users only)

The query depends only on the user's fixed pre-window `history`, identical for
every impression of that user - so retrieval only needs to run once per user,
not once per impression. Only users appearing in `val`/`test` need retrieval
computed (recall@K isn't reported on `train`).

In [11]:
user_topk = {}
coldstart_users = {}
for name in DATASETS:
    behaviors = feature_store[name]["behaviors"]
    history = feature_store[name]["history"]
    eval_user_ids = set(behaviors.loc[behaviors["split"].isin(["val", "test"]), "user_id"])
    history_eval = history[history["user_id"].isin(eval_user_ids)]
    lookup = title_by_id[name]

    topk_for_dataset = {}
    coldstart = set()
    for user_id, article_id_sequence in zip(history_eval["user_id"], history_eval["article_id_sequence"]):
        query_tokens = build_user_query_tokens(article_id_sequence, lookup)
        if not query_tokens:
            coldstart.add(user_id)
            continue
        topk_for_dataset[user_id] = top_k_candidates(query_tokens, name, k=TOPK_MAX)

    user_topk[name] = topk_for_dataset
    coldstart_users[name] = coldstart

{name: {"retrieved": len(user_topk[name]), "coldstart": len(coldstart_users[name])} for name in DATASETS}

{'ebnerd': {'retrieved': 1619, 'coldstart': 0},
 'mind': {'retrieved': 65173, 'coldstart': 1770}}

In [12]:
def test_user_retrieval_cache():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        eval_user_ids = set(behaviors.loc[behaviors["split"].isin(["val", "test"]), "user_id"])
        assert set(user_topk[name]).issubset(eval_user_ids)
        assert set(coldstart_users[name]).issubset(eval_user_ids)
        assert set(user_topk[name]) | set(coldstart_users[name]) == eval_user_ids

        history = feature_store[name]["history"]
        history_eval = history[history["user_id"].isin(eval_user_ids)]
        expected_coldstart = set(
            history_eval.loc[history_eval["article_id_sequence"].apply(len) == 0, "user_id"]
        )
        assert coldstart_users[name] == expected_coldstart, f"{name}: cold-start set mismatch"

        for aid_scores in list(user_topk[name].values())[:5]:
            assert 1 <= len(aid_scores) <= TOPK_MAX


test_user_retrieval_cache()
print("ok: retrieval cache covers exactly the val/test user population, cold-start counted independently")

ok: retrieval cache covers exactly the val/test user population, cold-start counted independently


## Recall@K evaluation

Fractional multi-relevant definition (`article_ids_clicked` is not always
singleton), macro-averaged over non-cold-start impressions per split.

In [13]:
def evaluate_recall(dataset: str, split: str, k_values: list[int]) -> dict:
    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors[behaviors["split"] == split]
    topk = user_topk[dataset]
    coldstart = coldstart_users[dataset]

    recalls = {k: [] for k in k_values}
    n_excluded = 0

    for user_id, clicked in zip(split_behaviors["user_id"], split_behaviors["article_ids_clicked"]):
        if user_id in coldstart:
            n_excluded += 1
            continue
        clicked = set(clicked)
        candidate_ids = [aid for aid, _ in topk[user_id]]
        for k in k_values:
            top_k_ids = set(candidate_ids[:k])
            recalls[k].append(len(clicked & top_k_ids) / len(clicked))

    n_evaluated = len(split_behaviors) - n_excluded
    return {
        "recall_at_k": {k: (sum(v) / len(v) if v else 0.0) for k, v in recalls.items()},
        "n_total": len(split_behaviors),
        "n_evaluated": n_evaluated,
        "n_excluded_coldstart": n_excluded,
    }


metrics = {name: {split: evaluate_recall(name, split, CANDIDATE_K_VALUES) for split in ["val", "test"]} for name in DATASETS}
metrics

{'ebnerd': {'val': {'recall_at_k': {50: 0.008807985907222548,
    100: 0.019377568995889608,
    200: 0.03508514386376982},
   'n_total': 3406,
   'n_evaluated': 3406,
   'n_excluded_coldstart': 0},
  'test': {'recall_at_k': {50: 0.0069805963085660195,
    100: 0.014611926171320398,
    200: 0.028522881782465315},
   'n_total': 25356,
   'n_evaluated': 25356,
   'n_excluded_coldstart': 0}},
 'mind': {'val': {'recall_at_k': {50: 0.011378536975187595,
    100: 0.021969983896216183,
    200: 0.034001383642461004},
   'n_total': 30270,
   'n_evaluated': 29498,
   'n_excluded_coldstart': 772},
  'test': {'recall_at_k': {50: 0.004477870838988998,
    100: 0.009243555348076197,
    200: 0.01611708192049476},
   'n_total': 73152,
   'n_evaluated': 70938,
   'n_excluded_coldstart': 2214}}}

In [14]:
def test_recall_at_k_monotonic():
    for name in DATASETS:
        for split in ["val", "test"]:
            m = metrics[name][split]
            r = m["recall_at_k"]
            assert r[50] <= r[100] + 1e-12 <= r[200] + 1e-12
            assert all(0.0 <= v <= 1.0 for v in r.values())
            assert m["n_evaluated"] + m["n_excluded_coldstart"] == m["n_total"]

    # verified separately that neither dataset has zero-click impressions
    for name in DATASETS:
        b = feature_store[name]["behaviors"]
        assert (b["article_ids_clicked"].apply(len) == 0).sum() == 0


test_recall_at_k_monotonic()
print("ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent")

ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent


## Persist BM25 outputs

Keyed by user (not impression) to avoid storing duplicate retrieval results
across a user's many impressions - mirrors Q1's `manifest.json` conventions.

In [15]:
def write_bm25_outputs(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    rows = [
        {
            "user_id": user_id,
            "dataset": dataset,
            "n_retrieved": len(aid_scores),
            "retrieved_article_ids": [aid for aid, _ in aid_scores],
            "retrieved_scores": [float(s) for _, s in aid_scores],
        }
        for user_id, aid_scores in user_topk[dataset].items()
    ]
    pd.DataFrame(rows).to_parquet(out_dir / "bm25_topk.parquet", index=False)

    bm25_metrics = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "k1": BM25_K1, "b": BM25_B,
            "recent_n_clicks": RECENT_N_CLICKS, "topk_max": TOPK_MAX,
        },
        "recall_at_k": {split: metrics[dataset][split]["recall_at_k"] for split in ["val", "test"]},
        "n_impressions": {
            split: {
                "total": metrics[dataset][split]["n_total"],
                "evaluated": metrics[dataset][split]["n_evaluated"],
                "excluded_coldstart": metrics[dataset][split]["n_excluded_coldstart"],
            }
            for split in ["val", "test"]
        },
        "scope": "val_test_users_only",
    }
    (out_dir / "bm25_metrics.json").write_text(json.dumps(bm25_metrics, indent=2))
    return out_dir


bm25_out_dirs = {name: write_bm25_outputs(name) for name in DATASETS}
bm25_out_dirs

{'ebnerd': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd'),
 'mind': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/mind')}

In [16]:
def test_bm25_outputs_roundtrip():
    for name in DATASETS:
        out_dir = bm25_out_dirs[name]
        topk_path = out_dir / "bm25_topk.parquet"
        metrics_path = out_dir / "bm25_metrics.json"
        assert topk_path.exists() and metrics_path.exists()

        reloaded_topk = pd.read_parquet(topk_path)
        assert set(reloaded_topk["user_id"]) == set(user_topk[name])
        for _, row in reloaded_topk.head(20).iterrows():
            assert len(row["retrieved_article_ids"]) == row["n_retrieved"]
            assert len(row["retrieved_scores"]) == row["n_retrieved"]

        reloaded_metrics = json.loads(metrics_path.read_text())
        for split in ["val", "test"]:
            for k in CANDIDATE_K_VALUES:
                expected = metrics[name][split]["recall_at_k"][k]
                actual = reloaded_metrics["recall_at_k"][split][str(k)]
                assert abs(expected - actual) < 1e-12


test_bm25_outputs_roundtrip()
print("ok: bm25_topk.parquet and bm25_metrics.json round-trip correctly for both datasets")

ok: bm25_topk.parquet and bm25_metrics.json round-trip correctly for both datasets
